# **Scraping Planos Nova Telecom**

_Página do provedor: https://www.rnova.com.br/_

In [1]:
# Bibliotecas
import pandas as pd
from bs4 import BeautifulSoup
from datetime import date

## **1. Extração**

In [3]:
# Obter lista de planos
texto_html = open("../../data/scraped/nova.html", encoding="utf-8")
soup = BeautifulSoup(texto_html, "html.parser")
container_planos = soup.find_all("ul", class_="glide__slides")[0]
planos = container_planos.find_all("li", class_="glide__slide")
print(f"{len(planos)} cartões de planos capturados.")

5 cartões de planos capturados.


In [4]:
# Extrair informações do plano
def raspar_plano(plano):
    cartao_velocidade = plano.find("div", class_="items-center")
    velocidade = cartao_velocidade.find_all("span")[0].text

    cartao_preco = plano.find("div", class_="items-baseline")
    valor_real = cartao_preco.find_all("span")[1].text
    valor_centavos = cartao_preco.find_all("span")[2].text
    preco = valor_real + valor_centavos

    bloco_svas = plano.find("ul")
    lista_svas = bloco_svas.find_all("li")
    lista_svas_textos = [i.get_text(strip=True) for i in lista_svas]
    svas = ", ".join(lista_svas_textos)

    return {"velocidade": velocidade,
            "preco": preco,
            "svas": svas}

# Iterando sobre a lista de planos
dados = []
for plano in planos:
    item = raspar_plano(plano)
    if item:
        dados.append(item)

df = pd.DataFrame(dados)
df

,velocidade,preco,svas
0,600,"99,90","Taxa de adesão e instalaçãográtis;, Nova Mais;..."
1,650,"114,00","Taxa de adesão e instalaçãográtis;, Nova Mais;..."
2,600,"114,00","Taxa de adesão e instalaçãográtis;, Globoplay*..."
3,680,"129,90","Taxa de adesão e instalaçãográtis;, Disney+;, ..."
4,680,"129,90","Taxa de adesão e instalaçãográtis;, HBO Max;, ..."


## **2. Tratamento**

In [5]:
df_tratado = df.copy()

# Data e provedor
df_tratado["DATA"] = date.today().strftime("%Y-%m-%d")
df_tratado["ISP"] = "Nova"

# Velocidade
df_tratado["VELOCIDADE"] = df_tratado["velocidade"]

# SVAs
df_tratado["SVAs"] = df_tratado["svas"]

# Preço: Substui vírgula por ponto
# Convert string em float
df_tratado["PRECO"] = (
    df_tratado["preco"]
    .str.replace(",", ".", regex=False)
    .astype(float)
)

#Seleção de colunas
df_tratado = df_tratado[["DATA", "ISP", "VELOCIDADE", "SVAs", "PRECO"]]

df_tratado

,DATA,ISP,VELOCIDADE,SVAs,PRECO
0,2026-05-12,Nova,600,"Taxa de adesão e instalaçãográtis;, Nova Mais;...",99.9
1,2026-05-12,Nova,650,"Taxa de adesão e instalaçãográtis;, Nova Mais;...",114.0
2,2026-05-12,Nova,600,"Taxa de adesão e instalaçãográtis;, Globoplay*...",114.0
3,2026-05-12,Nova,680,"Taxa de adesão e instalaçãográtis;, Disney+;, ...",129.9
4,2026-05-12,Nova,680,"Taxa de adesão e instalaçãográtis;, HBO Max;, ...",129.9


## **3. Carga**

In [6]:
# Salvando
provedor = "NOVA.csv"
nome_arq = df_tratado["DATA"].max() + "-" + provedor

# Na rede Valenet
pasta = r"\\fileserver\arquivos\Estrategia Comercial\2. POWER BI\ECM - Radar Mercado\ScrapingPlanos"
df_tratado.to_csv(fr"{pasta}\{nome_arq}", index=False)

# Neste Workspace
df_tratado.to_csv(f"../../data/flat/{provedor}", index=False)

print(f"🟢 {nome_arq} salvo!")

🟢 2026-05-12-NOVA.csv salvo!
